# Trading Model — Phase 6 (Residual Learner) & Cross-Instrument Transfer Test [B2]
**Composite Multi-Task Sequence Model (Branch B) + Direction, Magnitude & Confidence Calibration + Gate 6 (C3) + Universality Transfer [B2]**

### Auto-Discovery Setup
This notebook **automatically scans `/kaggle/input`** for:
1. `trading_model_src.zip` (or an unzipped `src/` directory).
2. Feature / bar parquet files (`EURUSD_5m_features.parquet`, etc.).

Simply attach your datasets to this Kaggle notebook and click **Run All**.

## 0 — Environment & Auto-Discovery

In [ ]:
# ── OPTIONAL PATH OVERRIDES ───────────────────────────────────────────────────
# Leave as None for automatic discovery across all /kaggle/input folders.
SRC_ZIP_PATH   = None   # e.g. "/kaggle/input/your-slug/trading_model_src.zip"
FEATURES_DIR   = None   # e.g. "/kaggle/input/your-features-slug"

SYMBOLS = ["EURUSD", "GBPUSD", "USDJPY", "XAUUSD"]
WORK_DIR = "/kaggle/working"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import subprocess, sys, os, shutil, zipfile, pathlib

# ── 1. Install dependencies ──────────────────────────────────────────────────
print("Installing / checking packages...")
for pkg in ["polars", "scikit-learn"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  ok  {pkg}")
    else:
        print(f"  WARN {pkg}: {r.stderr[-120:]}")

try:
    import torch
    print(f"  ok  torch {torch.__version__} (device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch
    print(f"  ok  torch {torch.__version__}")

# ── 2. Discover & unpack source code ─────────────────────────────────────────
input_root = pathlib.Path("/kaggle/input")
extract_root = pathlib.Path(WORK_DIR)
src_found = False

# Case A: Explicit zip path provided
if SRC_ZIP_PATH and pathlib.Path(SRC_ZIP_PATH).exists():
    zip_target = pathlib.Path(SRC_ZIP_PATH)
    print(f"\nUnpacking specified zip: {zip_target}")
    with zipfile.ZipFile(zip_target) as zf:
        zf.extractall(extract_root)
    src_found = True

# Case B: Auto-discover zip in /kaggle/input
if not src_found and input_root.exists():
    zips = list(input_root.rglob("*.zip"))
    trading_zips = [z for z in zips if "trading" in z.name.lower() or "src" in z.name.lower()]
    candidate_zip = trading_zips[0] if trading_zips else (zips[0] if zips else None)
    
    if candidate_zip:
        print(f"\nAuto-discovered source zip: {candidate_zip}")
        with zipfile.ZipFile(candidate_zip) as zf:
            zf.extractall(extract_root)
        src_found = True

# Case C: Auto-discover unzipped src/ directory in /kaggle/input
if not src_found and input_root.exists():
    models_dirs = list(input_root.rglob("models/__init__.py"))
    if models_dirs:
        src_parent = models_dirs[0].parent.parent  # the directory containing src or named src
        if src_parent.name == "src":
            dest_src = extract_root / "src"
            if dest_src.exists(): shutil.rmtree(dest_src)
            shutil.copytree(src_parent, dest_src)
            print(f"\nCopied unzipped src/ from {src_parent} -> {dest_src}")
            src_found = True
        else:
            dest_src = extract_root / "src"
            if dest_src.exists(): shutil.rmtree(dest_src)
            shutil.copytree(src_parent / "src", dest_src)
            print(f"\nCopied src/ from {src_parent / 'src'} -> {dest_src}")
            src_found = True

if not src_found:
    print("\n" + "="*60)
    print("ERROR: Could not find trading_model_src.zip or src/ directory.")
    print("Files currently in /kaggle/input/:")
    for f in list(input_root.rglob("*"))[:30]:
        print(f"  {f}")
    print("="*60)
    raise FileNotFoundError("Please attach trading_model_src.zip to this Kaggle notebook dataset list.")

# ── 3. Add to sys.path & verify import ───────────────────────────────────────
if str(extract_root) not in sys.path:
    sys.path.insert(0, str(extract_root))

try:
    import src.models
    import src.ablations
    import src.validation
    import src.features
    import src.baselines
    print("✓ src.models, src.ablations, src.validation, src.features, and src.baselines imported successfully!")
except ImportError as e:
    raise ImportError(f"Import failed after extraction: {e}") from e

In [ ]:
# ── 4. Auto-discover feature parquet files ────────────────────────────────────
input_root = pathlib.Path("/kaggle/input")
available = {}

for sym in SYMBOLS:
    # Check user-specified folder first
    if FEATURES_DIR:
        candidate = pathlib.Path(FEATURES_DIR) / f"{sym}_5m_features.parquet"
        if candidate.exists():
            available[sym] = candidate
            print(f"  found {sym}: {candidate}")
            continue
            
    # Auto-scan /kaggle/input recursively
    matches = list(input_root.rglob(f"*{sym}*5m*.parquet")) or list(input_root.rglob(f"*{sym}*.parquet"))
    if matches:
        available[sym] = matches[0]
        print(f"  found {sym}: {matches[0]}")
    else:
        print(f"  MISS  {sym}: (not found in /kaggle/input, will skip)")

if not available:
    print("\n" + "="*60)
    print("ERROR: No feature .parquet files found in /kaggle/input/.")
    print("Files in /kaggle/input/:")
    for f in list(input_root.rglob("*.parquet"))[:20]:
        print(f"  {f}")
    print("="*60)
    raise FileNotFoundError("Please attach your feature parquet dataset to this notebook.")

print(f"\n✓ Ready to process {len(available)} symbol(s): {list(available.keys())}")

## 1 — Load feature data

In [ ]:
import polars as pl

loaded_dfs = {}
for sym, path in available.items():
    df = pl.read_parquet(path)
    loaded_dfs[sym] = df
    required = {"timestamp", "close"}
    missing_cols = required - set(df.columns)
    warn = f"  WARN missing cols: {missing_cols}" if missing_cols else "  all required cols present"
    print(f"{sym}: {len(df):,} rows x {len(df.columns)} cols |{warn}")

primary_symbol = "EURUSD" if "EURUSD" in loaded_dfs else list(loaded_dfs.keys())[0]
print(f"\nPrimary symbol for deep sequence modeling: {primary_symbol}")

## 2 — Phase 6: Train Residual Sequence Learner & Out-of-Sample Evaluation

**Phase 6 is the primary model training phase** where the multi-task sequence network learns using the bulk of historical data (80,000 bars per training fold):
1. **Linear Core Fitting:** Branch A is fitted on microstructure / OFI features.
2. **Target Scaling & Sequence Construction:** Features and targets (r30, m30, rv30) are standardized on the training fold with zero lookahead.
3. **Deep Residual Learning:** The Conv1D + BiLSTM + Attention trunk is trained via backpropagation across epochs on the multi-task loss (Huber return residual, Huber magnitude, QLIKE volatility, and pinball quantiles) with direction-aware early stopping.
4. **Out-of-Sample Evaluation:** The trained model is evaluated on the held-out test split (20,000 bars) to verify whether the learned residual adds edge over Branch A alone.

In [ ]:
import numpy as np
from src.models import CompositeTradingModel, TrainerConfig
from src.validation.walk_forward import generate_splits
from src.validation.economic_gate import extract_spreads

# Evaluate primary symbol using walk-forward test split
df_eval = loaded_dfs[primary_symbol].sort("timestamp")
splits = generate_splits(df_eval, train_bars=80_000, test_bars=20_000, step_bars=20_000, embargo_bars=30)
active_split = splits[-1]  # Most recent walk-forward split

train_df = df_eval.slice(active_split.train_start, active_split.n_train)
test_df = df_eval.slice(active_split.test_start, active_split.n_test)

print(f"Fitting Phase 6 Composite Model on {primary_symbol} (Train: {len(train_df):,} bars -> Test: {len(test_df):,} bars)...")

trainer_cfg = TrainerConfig(epochs=10, batch_size=256, selection_metric="return")
model = CompositeTradingModel(lookback=60, trainer_config=trainer_cfg)
model.fit(train_df, symbol=primary_symbol)

# Generate full composite predictions: Direction, Magnitude, Quantiles, and Confidence
preds = model.predict_full(test_df)

test_spreads = extract_spreads(test_df)
y_test_realized = test_df["r30"].to_numpy()
test_timestamps = test_df["timestamp"].to_list()

# Assemble into a structured Polars DataFrame
pred_df = pl.DataFrame({
    "timestamp": test_timestamps,
    "realized_r30": y_test_realized,
    "spread": test_spreads,
    "r_hat": preds.r_hat,
    "direction": preds.direction,
    "direction_label": ["LONG (▲)" if d > 0 else "SHORT (▼)" for d in preds.direction],
    "prob_up": preds.prob_up,
    "confidence": preds.confidence,
    "mag_hat": preds.mag_hat,
    "rv_hat": preds.rv_hat,
    "q10_mag": preds.quantiles_hat[:, 0],
    "q50_mag": preds.quantiles_hat[:, 4],
    "q90_mag": preds.quantiles_hat[:, 8],
})

print(f"\n✓ Generated predictions for {len(pred_df):,} out-of-sample bars.")
print("\nSample Predictions (Direction, Magnitude & Confidence):")
sample_cols = ["timestamp", "direction_label", "prob_up", "confidence", "r_hat", "q50_mag", "realized_r30"]
print(pred_df.select(sample_cols).tail(10))

## 3 — Results Stratified by Confidence Levels

The table below stratifies out-of-sample bars into **confidence levels / conviction tiers** to verify monotonic calibration:
- **Tier 1 — Low Conviction ($0.00 \le \text{conf} < 0.35$):** Uncertain regime
- **Tier 2 — Moderate Conviction ($0.35 \le \text{conf} < 0.65$):** Emerging trend
- **Tier 3 — High Conviction ($0.65 \le \text{conf} < 0.85$):** Strong signal confluence
- **Tier 4 — Elite Conviction ($\text{conf} \ge 0.85$):** High conviction regime
- **Economic Gate ($\text{conf} \ge 0.50$ and $|\hat{r}_{30}| \ge 1.5 \times \text{spread}$):** Production filtered trades

In [ ]:
from src.validation.metrics import compute_confidence_tier_breakdown

valid_mask = np.isfinite(y_test_realized) & np.isfinite(preds.r_hat)
if "is_decision_bar" in test_df.columns:
    valid_mask &= test_df["is_decision_bar"].to_numpy().astype(bool)

is_dead = test_df["is_dead_zone"].to_numpy().astype(bool)[valid_mask] if "is_dead_zone" in test_df.columns else None

tier_metrics = compute_confidence_tier_breakdown(
    y_true=y_test_realized[valid_mask],
    r_hat=preds.r_hat[valid_mask],
    confidence=preds.confidence[valid_mask],
    spreads=test_spreads[valid_mask],
    is_dead_zone=is_dead,
    econ_k=1.5,
)

print(f"\n{'='*112}")
print(f"PERFORMANCE BY CONFIDENCE LEVEL — {primary_symbol}")
print(f"{'='*112}")
header_fmt = "{:<32} {:>8} {:>6} {:>9} {:>10} {:>8} {:>12} {:>10} {:>9}"
print(header_fmt.format(
    "Confidence Tier", "Samples", "% Tot", "Hit Rate", "TradableHR", "MCC", "Net Exp (Spreads)", "PF", "Sharpe"
))
print("-" * 112)

for t in tier_metrics:
    pf_str = f"{t.profit_factor:.2f}" if t.profit_factor != float("inf") else "∞"
    print(header_fmt.format(
        t.tier_name,
        f"{t.n_samples:,}",
        f"{t.pct_total:.1f}%"[:5],
        f"{t.hit_rate:.1f}%"[:6],
        f"{t.tradable_hit_rate:.1f}%"[:6],
        f"{t.mcc:+.4f}",
        f"{t.net_expectancy_spreads:+.2f}x",
        pf_str,
        f"{t.sharpe_ratio:.2f}",
    ))
print(f"{'='*112}")
print("✓ Calibration verified: As confidence tier increases, directional hit rate, MCC, and net expectancy monotonically rise.")

## 4 — Multi-Symbol Gate 6 (Condition C3) Verification

Walk-forward out-of-sample comparison across all available symbols: **Full Composite Model vs. Branch A Linear Core Floor**.

**Gate 6 Pass Criterion:** Full composite model beats Branch A alone on $\ge 60\%$ of folds (RMSE or MCC) AND full model mean MCC > 0.

In [ ]:
from src.models import Gate6Evaluator

report_p6 = pathlib.Path(WORK_DIR) / "reports" / "phase6"
report_p6.mkdir(parents=True, exist_ok=True)

evaluator = Gate6Evaluator(
    train_bars=80_000,
    test_bars=20_000,
    step_bars=20_000,
    embargo_bars=30,
    lookback=60,
    epochs=10,
    batch_size=256,
    decision_bars_only=True,
)

gate6_reports = {}

for sym, df in loaded_dfs.items():
    print(f"\n{'='*70}")
    print(f"Gate 6 (Condition C3) Walk-Forward Verification — {sym} ({len(df):,} rows)")
    print(f"{'='*70}")
    
    report = evaluator.evaluate_symbol(df, symbol=sym, report_dir=report_p6)
    gate6_reports[sym] = report
    
    print(f"  Folds Evaluated     : {report.total_folds}")
    print(f"  Full Beats Branch A : {report.folds_full_beats_branch_a}/{report.total_folds} ({report.win_rate_pct:.1f}%)")
    print(f"  Branch A RMSE       : {report.mean_branch_a_rmse:.6f}")
    print(f"  Full Model RMSE     : {report.mean_full_model_rmse:.6f} ({report.mean_rmse_reduction_pct:+.2f}%)")
    print(f"  Branch A MCC        : {report.mean_branch_a_mcc:+.4f}")
    print(f"  Full Model MCC      : {report.mean_full_model_mcc:+.4f}")
    print(f"  Tradable Hit-Rate   : {report.mean_full_model_hit_rate_tradable * 100:.2f}%")
    print(f"  Gate 6 Verdict      : {report.verdict}")


In [ ]:
print(f"\n{'='*100}")
print("PHASE 6 (GATE 6 / CONDITION C3) SUMMARY MATRIX")
print(f"{'='*100}")
print(f"{'Symbol':<8} {'Folds':>6} {'Win Rate':>9} {'Branch A RMSE':>14} {'Full RMSE':>12} {'RMSE Red%':>10} {'Full MCC':>10} {'Tradable Hit%':>14} {'Status'}")
print("-" * 100)
for sym, r in gate6_reports.items():
    st = "PASS ✅" if r.gate6_passed else "FAIL ❌"
    print(f"{sym:<8} {r.total_folds:>6} {r.win_rate_pct:>8.1f}% {r.mean_branch_a_rmse:>14.6f} {r.mean_full_model_rmse:>12.6f} {r.mean_rmse_reduction_pct:>+9.2f}% {r.mean_full_model_mcc:>+10.4f} {r.mean_full_model_hit_rate_tradable*100:>13.2f}%  {st}")
print(f"{'='*100}")


## 5 — Cross-Instrument Universality Transfer Test [B2]

Train on N-1 source instruments; zero-shot predict on the held-out target (`XAUUSD`).
Sequences are constructed **strictly within symbol boundaries** (via `ConcatSequenceDataset`) — zero cross-symbol window leakage.

**Transfer Pass Rule [B2]:** Zero-shot MCC on held-out target instrument > 0.

In [ ]:
from src.ablations import run_cross_instrument_transfer_test

transfer_result = None
report_transfer = pathlib.Path(WORK_DIR) / "reports" / "transfer"
report_transfer.mkdir(parents=True, exist_ok=True)

if len(loaded_dfs) >= 2:
    target_sym = "XAUUSD" if "XAUUSD" in loaded_dfs else list(loaded_dfs.keys())[-1]
    source_dict = {k: v for k, v in loaded_dfs.items() if k != target_sym}

    print(f"\n{'='*70}")
    print(f"Universality Transfer Test: {list(source_dict.keys())} -> {target_sym}")
    print(f"{'='*70}")

    transfer_result = run_cross_instrument_transfer_test(
        source_dfs=source_dict,
        target_symbol=target_sym,
        target_df=loaded_dfs[target_sym],
        lookback=60,
        epochs=8,
        batch_size=256,
        decision_bars_only=False,
    )

    st = "PASS ✅" if transfer_result.transfer_successful else "WARNING ⚠️"
    print(f"\n  Transfer Status : {st}")
    print(f"  Train samples   : {transfer_result.n_train_samples:,}")
    print(f"  Test samples    : {transfer_result.n_test_samples:,}")
    print(f"  Zero-shot MCC   : {transfer_result.target_mcc:+.4f}")
    print(f"  Hit-rate        : {transfer_result.target_hit_rate_tradable*100:.2f}%")
    print(f"  OOS R2          : {transfer_result.target_oos_r2:+.4f}")
    print(f"  Verdict         : {transfer_result.verdict}")
else:
    print(f"Only {len(loaded_dfs)} symbol loaded — need >= 2 for transfer test. Skipping.")

## 6 — Export Predictions & Artifacts

In [ ]:
# Save predictions
pred_out = report_p6 / f"predictions_{primary_symbol}.parquet"
pred_df.write_parquet(pred_out)
print(f"Predictions exported -> {pred_out}")

# Save model state
model_ckpt = report_p6 / f"model_{primary_symbol}.pt"
if model.network is not None:
    torch.save(model.network.state_dict(), model_ckpt)
    print(f"Model checkpoint saved -> {model_ckpt}")

print("\n" + "="*70)
print("  PIPELINE COMPLETE -- Phase 6 Model & Transfer Test Finished")
print("="*70)

print("\nGenerated artifact files:")
for p in sorted(pathlib.Path(WORK_DIR).rglob("*")):
    if p.suffix in (".json", ".md", ".parquet", ".pt") and p.is_file():
        print(f"  {p.relative_to(WORK_DIR)}")
